In [1]:
import os
import pandas as pd
import polars as pl
import snapatac2 as snap
import scanpy as sc
import anndata as ad
import numpy as np

In [2]:
work_dir = '/hdd/jupyter/brad/snapatac/motor-pathway_multiome/motor-pathway_multiome_cellbender_cluster-snrna-cr/'
script_name = 'snapatac_preprocess'
work_dir = os.path.join(work_dir, script_name)

bw_dir = os.path.join(work_dir, 'bw')
data_sub_fname = os.path.join(work_dir,  "subset.h5ad")

In [3]:
data_sub = snap.read(filename = data_sub_fname)

In [4]:
#data_sub.close()

In [5]:
split_adata = {}
for position in data_sub.obs['position'].unique():
    print(position)
    obs = data_sub.obs['position']
    cells_to_use = obs == position
    split_adata[position] = data_sub.subset(obs_indices=cells_to_use, inplace=False)
   
    obs = split_adata[position].obs['cluster']
    obs_num = obs.value_counts()
    obs_num_keep = obs_num[obs_num >= 10]
    obs_keep = obs.isin(obs_num_keep.index)
    split_adata[position] = split_adata[position][obs_keep]

/tmp/ipykernel_3194734/980122236.py:2: DeprecationWarning: `Series._import_from_c` is deprecated. use _import_arrow_from_c; if you are using an extension, please compile it with latest 'pyo3-polars'
  for position in data_sub.obs['position'].unique():
/tmp/ipykernel_3194734/980122236.py:4: DeprecationWarning: `Series._import_from_c` is deprecated. use _import_arrow_from_c; if you are using an extension, please compile it with latest 'pyo3-polars'
  obs = data_sub.obs['position']


ra


/tmp/ipykernel_3194734/980122236.py:6: DeprecationWarning: `Series._import_from_c` is deprecated. use _import_arrow_from_c; if you are using an extension, please compile it with latest 'pyo3-polars'
  split_adata[position] = data_sub.subset(obs_indices=cells_to_use, inplace=False)
/tmp/ipykernel_3194734/980122236.py:6: DeprecationWarning: `Series._import_from_c` is deprecated. use _import_arrow_from_c; if you are using an extension, please compile it with latest 'pyo3-polars'
  split_adata[position] = data_sub.subset(obs_indices=cells_to_use, inplace=False)


arco


/tmp/ipykernel_3194734/980122236.py:4: DeprecationWarning: `Series._import_from_c` is deprecated. use _import_arrow_from_c; if you are using an extension, please compile it with latest 'pyo3-polars'
  obs = data_sub.obs['position']
/tmp/ipykernel_3194734/980122236.py:6: DeprecationWarning: `Series._import_from_c` is deprecated. use _import_arrow_from_c; if you are using an extension, please compile it with latest 'pyo3-polars'
  split_adata[position] = data_sub.subset(obs_indices=cells_to_use, inplace=False)
/tmp/ipykernel_3194734/980122236.py:6: DeprecationWarning: `Series._import_from_c` is deprecated. use _import_arrow_from_c; if you are using an extension, please compile it with latest 'pyo3-polars'
  split_adata[position] = data_sub.subset(obs_indices=cells_to_use, inplace=False)


hvc


/tmp/ipykernel_3194734/980122236.py:4: DeprecationWarning: `Series._import_from_c` is deprecated. use _import_arrow_from_c; if you are using an extension, please compile it with latest 'pyo3-polars'
  obs = data_sub.obs['position']
/tmp/ipykernel_3194734/980122236.py:6: DeprecationWarning: `Series._import_from_c` is deprecated. use _import_arrow_from_c; if you are using an extension, please compile it with latest 'pyo3-polars'
  split_adata[position] = data_sub.subset(obs_indices=cells_to_use, inplace=False)
/tmp/ipykernel_3194734/980122236.py:6: DeprecationWarning: `Series._import_from_c` is deprecated. use _import_arrow_from_c; if you are using an extension, please compile it with latest 'pyo3-polars'
  split_adata[position] = data_sub.subset(obs_indices=cells_to_use, inplace=False)


nc


/tmp/ipykernel_3194734/980122236.py:4: DeprecationWarning: `Series._import_from_c` is deprecated. use _import_arrow_from_c; if you are using an extension, please compile it with latest 'pyo3-polars'
  obs = data_sub.obs['position']
/tmp/ipykernel_3194734/980122236.py:6: DeprecationWarning: `Series._import_from_c` is deprecated. use _import_arrow_from_c; if you are using an extension, please compile it with latest 'pyo3-polars'
  split_adata[position] = data_sub.subset(obs_indices=cells_to_use, inplace=False)
/tmp/ipykernel_3194734/980122236.py:6: DeprecationWarning: `Series._import_from_c` is deprecated. use _import_arrow_from_c; if you are using an extension, please compile it with latest 'pyo3-polars'
  split_adata[position] = data_sub.subset(obs_indices=cells_to_use, inplace=False)


In [6]:
split_adata['hvc'].obs['cluster'].value_counts()

cluster
Glut-HVC-1     1257
Glut-Nido-3     642
GABA-1-1        333
Glut-Pre-2      322
Glut-NC-2       316
Glut-Pre-3      313
Glut-NC-4       310
Glut-NC-1       256
Glut-HVC-2      244
Oligo           163
Glut-Pre-1      146
GABA-1-2        131
Epen            116
Astro-2         111
GABA-3          101
GABA-2-1         98
OPC              96
Astro-1          52
GABA-Pre         43
Glut-NC-3        41
Micro            38
GABA-4-1         19
GABA-5           16
GABA-6           15
Name: count, dtype: int64

In [ ]:
bin_size = 50
smooth_base = 0
counting_strategy = "insertion"
bw_subdir = os.path.join(bw_dir, f'{counting_strategy}_bin{bin_size}-smooth{smooth_base}')
os.makedirs(bw_subdir, exist_ok=True)

snap.ex.export_coverage(data_sub,
                        counting_strategy = counting_strategy, 
                        smooth_base = smooth_base,
                        groupby='cluster', 
                        suffix=f'-{counting_strategy}-bin{bin_size}-smooth{smooth_base}.bw',
                        out_dir=bw_subdir, 
                        bin_size=bin_size)

2026-02-04 22:04:26 - INFO - Exporting fragments...
2026-02-04 22:06:42 - INFO - Computing coverage...


In [ ]:
bin_size = 10
smooth_base = 20
counting_strategy = "insertion"
bw_subdir = os.path.join(bw_dir, f'{counting_strategy}_bin{bin_size}-smooth{smooth_base}')
os.makedirs(bw_subdir, exist_ok=True)

for key, adata_cur in split_adata.items():
    bw_subdir_cur = os.path.join(bw_subdir, key)
    snap.ex.export_coverage(adata_cur, 
                        counting_strategy = counting_strategy, 
                        smooth_base = smooth_base,
                        groupby='cluster', 
                        suffix=f'-{counting_strategy}-bin{bin_size}-smooth{smooth_base}.bw',
                        out_dir=bw_subdir_cur, 
                        bin_size=bin_size)